# Mrk 509: a single-object distance duality test

Mrk 509 is the only quasar with both a SARM angular diameter distance
(Li et al. 2025, Table 2) and UV and X-ray fluxes in the UV--X-ray catalogue
(Lusso et al. 2020, where it is one of the 13 local AGN). Here we combine the
two to measure $\eta(z) = d_{\rm L}(z) / [(1+z)^2 d_{\rm A}(z)]$.

## Import packages

In [1]:
import numpy as np
import pandas as pd

## Luminosity distance
[Lusso et al. (2020)](https://vizier.cds.unistra.fr/viz-bin/VizieR?-source=J/A+A/642/A150) gives a distance modulus for each quasar, with the UV--X-ray intercept cross-calibrated on Pantheon supernovae assuming $H_0 = 70$ km/s/Mpc.

In [2]:
# load the data
columns = ['name', 'ra', 'dec', 'z', 'logFUV', 'e_logFUV', 'logFX', 'e_logFX',
           'group', 'gammax', 'e_gammax', 'DM', 'e_DM']

lusso = pd.read_csv('data/lusso2020_table3.dat', sep=r'\s+', names=columns)

mrk509 = lusso.set_index('name').loc['MRK509']

# get the redshift
z = mrk509['z']

In [3]:
# compute dL from the distance modulus mu given in the catalogue
dL = 10**(mrk509['DM']/5 - 5)  # Mpc

# and the uncertainty; do it in log space for greater accuracy when propagating to eta
# we include 0.2 dex of intrinsic scatter in the UV-X relation, and its slope, cf. Lusso et al. 2008.08586,
delta = 0.2  # dex
gamma = 0.6

# uncertainty from the distance modulus and the intrinsic scatter added in quadrature
sigma_ln_dL = np.sqrt((np.log(10)/5 * mrk509['e_DM'])**2 + np.log(10) * delta / (2 * (1 - gamma))**2)

dL_up = dL * np.exp(sigma_ln_dL) - dL
dL_down = dL - dL * np.exp(-sigma_ln_dL)

In [4]:
print(r'Mrk509 has a luminosity distance of ${:.2f}^{{+{:.1f}}}_{{-{:.1f}}}$ Mpc.'.format(dL, dL_up, dL_down))

Mrk509 has a luminosity distance of $136.49^{+182.8}_{-78.1}$ Mpc.


## Angular diameter distance
[Li et al. (2025)](https://arxiv.org/abs/2502.18856) use SARM to find $\log(d_A/{\rm Mpc}) = 2.41^{+0.22}_{-0.23}$.

In [5]:
log_dA = 2.41

dA = 10**log_dA # Mpc

# and the uncertainty; symmetrised for simplicity
log_sigma_average = (0.22+0.23)/2
sigma_ln_dA = np.log(10) * log_sigma_average

dA_up = dA * np.exp(sigma_ln_dA) - dA
dA_down = dA - dA * np.exp(-sigma_ln_dA)

In [6]:
print(r'Mrk509 has an angular diameter distance of ${:.2f}^{{+{:.1f}}}_{{-{:.1f}}}$ Mpc.'.format(dA, dA_up, dA_down))

Mrk509 has an angular diameter distance of $257.04^{+174.5}_{-103.9}$ Mpc.


## Measure $\eta(z)$

In [7]:
eta = dL/(((1+z)**2)* dA)

# and the uncertainty
sigma_ln_eta = np.sqrt(sigma_ln_dL**2 + sigma_ln_dA**2)

eta_up = eta * np.exp(sigma_ln_eta) - eta
eta_down = eta - eta * np.exp(-sigma_ln_eta)

# expectation with no DDR violation is eta = 1
sigmas = (1 - eta)/ eta_up

In [9]:
print(r'The distance duality relation measured from Mrk509 is $\eta = {:.2f}^{{+{:.1f}}}_{{-{:.1f}}}$.'.format(eta, eta_up, eta_down))
print(r'This is ${:.1f} \sigma$ from no DDR violation ($\eta = 1$).'.format(sigmas))

The distance duality relation measured from Mrk509 is $\eta = 0.50^{+0.8}_{-0.3}$.
This is $0.6 \sigma$ from no DDR violation ($\eta = 1$).


## Rescale with different $H_0$ values

In [10]:
 # Lusso et al. calibrate on supernovae assuming H0 = 70; dL scales as 1/H0, whilst the SARM dA is geometric and does not
H0_lusso = 70.0
H0_forecast = 67.36

eta_rescaled = eta * H0_lusso / H0_forecast

eta_up_rescaled = eta_rescaled * np.exp(sigma_ln_eta) - eta_rescaled
eta_down_rescaled = eta_rescaled - eta * np.exp(-sigma_ln_eta)

# expectation with no DDR violation is eta = 1
sigmas_rescaled = (1 - eta_rescaled)/ eta_up_rescaled

print(r'The distance duality relation measured from Mrk509 with $H_0 = {}$ km/s/Mpc is $\eta = {:.2f}^{{+{:.1f}}}_{{-{:.1f}}}$.'.format(H0_forecast, eta_rescaled, eta_up_rescaled, eta_down_rescaled))
print(r'This is ${:.1f} \sigma$ from no DDR violation ($\eta = 1$).'.format(sigmas_rescaled))

The distance duality relation measured from Mrk509 with $H_0 = 67.36$ km/s/Mpc is $\eta = 0.52^{+0.9}_{-0.3}$.
This is $0.6 \sigma$ from no DDR violation ($\eta = 1$).
